In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    count,
    sum,
)
from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    write_clickhouse,
)
from etl.transformations.facts.common import add_date_key

In [2]:
def build_daily_farm_quality_metrics(
    fact_harvests_df: DataFrame,
) -> DataFrame:
    """
    Aggregate harvest yield by farm, day, and quality grade.
    """

    return (
        fact_harvests_df.groupBy(
            "harvest_date",
            "farm_key",
            "farm_id",
            "quality_grade_id",
        )
        .agg(
            sum("weight_kg").alias(
                "total_yield_kg",
            ),
            count("*")
            .cast("integer")
            .alias(
                "harvest_count",
            ),
        )
        .withColumnRenamed(
            "harvest_date",
            "metric_date",
        )
    )


def transform_daily_farm_quality_metrics(
    fact_harvests_df: DataFrame,
    dim_date_df: DataFrame,
) -> DataFrame:
    """
    Build daily farm quality metrics.
    """

    metrics_df = build_daily_farm_quality_metrics(
        fact_harvests_df,
    )

    metrics_df = add_date_key(
        metrics_df,
        dim_date_df,
        "metric_date",
    )

    return metrics_df

    # .select(
    #     "metric_date",
    #     "date_key",
    #     "farm_key",
    #     "farm_id",
    #     "quality_grade_id",
    #     "total_yield_kg",
    #     "harvest_count",
    # )


def main():
    spark = create_spark(
        "fact_daily_farm_quality_metrics",
    )

    try:
        fact_harvests_df = read_clickhouse(
            spark,
            "fact_harvests",
        )

        dim_date_df = read_clickhouse(
            spark,
            "dim_date",
        )

        daily_quality_metrics_df = transform_daily_farm_quality_metrics(
            fact_harvests_df,
            dim_date_df,
        )

        write_clickhouse(
            daily_quality_metrics_df,
            "fact_daily_farm_quality_metrics",
        )

    finally:
        spark.stop()


if __name__ == "__main__":
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 10:11:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:11:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
